# Как улучшить модель

ФПМИ.Будущее · ИИ · Прекурс · Занятие 10

Бейзлайн пишется за десять минут. Дальше начинается работа, за которую дают баллы: превратить «оно как-то предсказывает» в «оно предсказывает хорошо».

Сегодня по шагам улучшаем одно решение и каждый раз проверяем, что улучшение настоящее, а не показалось.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

plt.style.use("dark_background")
plt.rcParams["figure.facecolor"] = "#0b1020"
plt.rcParams["axes.facecolor"] = "#111827"
plt.rcParams["savefig.facecolor"] = "#0b1020"
plt.rcParams["axes.grid"] = True
plt.rcParams["grid.alpha"] = 0.25

rng = np.random.default_rng(10)

## 1. Данные

Задача — предсказать стоимость аренды квартиры. Такие данные бывают «грязными»: где-то пропуски, где-то текстовая колонка, где-то выбросы. Соберём похожий датасет сами, чтобы всё было воспроизводимо.

In [ ]:
n = 1200

district = rng.choice(["центр", "север", "юг", "запад"], size=n, p=[0.2, 0.3, 0.3, 0.2])
district_bonus = pd.Series(district).map({"центр": 45, "север": 5, "юг": 0, "запад": 15}).to_numpy()

# У каждого дома своя надбавка: ремонт, двор, соседи.
building = rng.integers(0, 60, n)
building_bonus = rng.normal(0, 7, 60)[building]

area = rng.uniform(28, 120, n).round(1)
rooms = np.clip((area // 30 + rng.integers(0, 2, n)), 1, 5).astype(int)
floor = rng.integers(1, 26, n)
year = rng.integers(1960, 2024, n)
metro_min = rng.uniform(2, 35, n).round(1)

price = (
    14 * rooms
    + 0.55 * area
    + 1.9 * (area / rooms)          # просторность комнат
    + district_bonus
    + building_bonus
    + 0.25 * (year - 1960)
    - 0.9 * metro_min
    - 11 * (floor == 1)             # первый этаж дешевле
    + rng.normal(0, 6, n)
)

data = pd.DataFrame({
    "area": area, "rooms": rooms, "floor": floor,
    "year": year.astype(float), "metro_min": metro_min,
    "district": district, "building": building,
    "price": price.round(1),
})

# Пропуски в годе постройки — как в реальных выгрузках.
data.loc[rng.choice(n, size=90, replace=False), "year"] = np.nan

data.head()

In [ ]:
print("размер:", data.shape)
print("пропуски:")
print(data.isna().sum())

## 2. Бейзлайн

Берём только числовые признаки, пропуски заполняем медианой, обучаем линейную регрессию. Качество меряем через RMSE: корень из среднего квадрата ошибки, в тех же единицах, что и цена. Чем меньше, тем лучше.

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.pipeline import make_pipeline
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error

NUM = ["area", "rooms", "floor", "year", "metro_min"]

X = data.drop(columns="price")
y = data["price"]

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.25, random_state=42)

baseline = make_pipeline(SimpleImputer(strategy="median"), LinearRegression())
baseline.fit(X_train[NUM], y_train)

def rmse(y_true, y_pred):
    return np.sqrt(mean_squared_error(y_true, y_pred))

print("RMSE бейзлайна на тесте:", round(rmse(y_test, baseline.predict(X_test[NUM])), 3))

## 3. Одного разбиения мало

Поменяем только `random_state` — данные и модель те же. Если качество заметно скачет, то и разница между двумя моделями в пределах этого разброса ничего не значит.

In [ ]:
scores = []
for seed in range(8):
    Xtr, Xte, ytr, yte = train_test_split(X, y, test_size=0.25, random_state=seed)
    model = make_pipeline(SimpleImputer(strategy="median"), LinearRegression()).fit(Xtr[NUM], ytr)
    scores.append(rmse(yte, model.predict(Xte[NUM])))

scores = np.array(scores)
print("RMSE по разбиениям:", scores.round(2))
print(f"разброс: от {scores.min():.2f} до {scores.max():.2f}")

## 4. Кросс-валидация

Делим данные на $K$ частей. Каждая часть по очереди работает валидацией, остальные — обучением. Модель обучается $K$ раз, а качеством считаем среднее. Заодно получаем разброс: он показывает, какая разница вообще заслуживает внимания.

In [ ]:
from sklearn.model_selection import KFold, cross_val_score

cv = KFold(n_splits=5, shuffle=True, random_state=0)

def cv_rmse(model, features=NUM, X=X, y=y):
    scores = -cross_val_score(model, X[features], y, cv=cv, scoring="neg_root_mean_squared_error")
    return scores.mean(), scores.std()

mean, std = cv_rmse(make_pipeline(SimpleImputer(strategy="median"), LinearRegression()))
print(f"бейзлайн: RMSE = {mean:.2f} ± {std:.2f}")

Дальше сравниваем всё именно так: среднее по фолдам и разброс. Улучшение засчитываем, если оно заметно больше разброса.

## 5. Признаки решают больше, чем модель

### Категории

`district` — текст, модель его не видит. One-hot превращает колонку в несколько признаков-индикаторов: по одному на район.

In [ ]:
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder

CAT = ["district"]

prep = ColumnTransformer([
    ("num", SimpleImputer(strategy="median"), NUM),
    ("cat", OneHotEncoder(handle_unknown="ignore"), CAT),
])

mean, std = cv_rmse(make_pipeline(prep, LinearRegression()), features=NUM + CAT)
print(f"+ район: RMSE = {mean:.2f} ± {std:.2f}")

### Новые признаки из старых

Модель видит только то, что ей дали. Площадь и число комнат есть, а площади на комнату нет — хотя «сорок метров на одну комнату» и «сорок метров на три» это разные квартиры.

In [ ]:
def add_features(df):
    df = df.copy()
    df["area_per_room"] = df["area"] / df["rooms"]      # просторность
    df["is_first_floor"] = (df["floor"] == 1).astype(int)
    return df

X_fe = add_features(X)
NUM_FE = NUM + ["area_per_room", "is_first_floor"]

prep_fe = ColumnTransformer([
    ("num", SimpleImputer(strategy="median"), NUM_FE),
    ("cat", OneHotEncoder(handle_unknown="ignore"), CAT),
])

mean, std = cv_rmse(make_pipeline(prep_fe, LinearRegression()), features=NUM_FE + CAT, X=X_fe)
print(f"+ новые признаки: RMSE = {mean:.2f} ± {std:.2f}")

Оба признака уже «лежали» в данных: площадь на комнату — это площадь, делённая на комнаты, а первый этаж — это этаж. Но линейная модель складывает признаки с весами и сама такое не соберёт, поэтому подсказку нужно дать руками.

## 6. Масштабирование

Линейным моделям с регуляризацией и методам, считающим расстояния, важен масштаб признаков: площадь в десятках, год в тысячах. Деревьям всё равно — они сравнивают значения с порогом, а не складывают их.

In [ ]:
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import Ridge
from sklearn.tree import DecisionTreeRegressor

ridge_plain = make_pipeline(prep_fe, Ridge(alpha=10))
ridge_scaled = make_pipeline(prep_fe, StandardScaler(with_mean=False), Ridge(alpha=10))
tree_plain = make_pipeline(prep_fe, DecisionTreeRegressor(max_depth=8, random_state=0))
tree_scaled = make_pipeline(prep_fe, StandardScaler(with_mean=False), DecisionTreeRegressor(max_depth=8, random_state=0))

for name, model in [("Ridge без масштабирования", ridge_plain),
                    ("Ridge с масштабированием", ridge_scaled),
                    ("дерево без масштабирования", tree_plain),
                    ("дерево с масштабированием", tree_scaled)]:
    mean, std = cv_rmse(model, features=NUM_FE + CAT, X=X_fe)
    print(f"{name:32s} RMSE = {mean:.2f} ± {std:.2f}")

## 7. Гиперпараметры подбираем на валидации

Перебираем глубину дерева и смотрим две кривые: ошибку на обучении и ошибку на валидации. Там, где обучающая ошибка продолжает падать, а валидационная уже растёт, начинается переобучение.

In [ ]:
from sklearn.model_selection import cross_validate

depths = [2, 3, 4, 6, 8, 10, 14, 18, None]
train_curve, val_curve = [], []

for depth in depths:
    model = make_pipeline(prep_fe, DecisionTreeRegressor(max_depth=depth, random_state=0))
    res = cross_validate(model, X_fe[NUM_FE + CAT], y, cv=cv,
                         scoring="neg_root_mean_squared_error", return_train_score=True)
    train_curve.append(-res["train_score"].mean())
    val_curve.append(-res["test_score"].mean())

for depth, tr, va in zip(depths, train_curve, val_curve):
    print(f"depth={str(depth):>4}  train={tr:6.2f}  val={va:6.2f}")

In [ ]:
xs = range(len(depths))
plt.figure(figsize=(8, 4.2), layout="constrained")
plt.plot(xs, train_curve, marker="o", color="#60A5FA", label="train")
plt.plot(xs, val_curve, marker="s", color="#F59E0B", label="валидация")
plt.xticks(list(xs), [str(d) for d in depths])
plt.xlabel("максимальная глубина дерева")
plt.ylabel("RMSE")
plt.title("Чем глубже дерево, тем лучше на train и хуже на валидации")
plt.legend()
plt.show()

Правила простые: менять по одному параметру, фиксировать `random_state`, сравнивать с разбросом. И помнить, что если долго перебирать на одной валидации, начнёшь подгоняться уже под неё.

## 8. Утечка данных

Утечка — это когда в признаки попадает то, чего в момент предсказания знать неоткуда. Валидация показывает прекрасный результат, а на приватной части случается провал.

### Признак из будущего

В выгрузке часто есть залог. В объявлениях он обычно равен месячной аренде — то есть считается из той самой цены, которую мы предсказываем.

In [ ]:
X_leak = X_fe.copy()
X_leak["deposit"] = (y + rng.normal(0, 2, n)).round(1)     # так делать нельзя

prep_leak = ColumnTransformer([
    ("num", SimpleImputer(strategy="median"), NUM_FE + ["deposit"]),
    ("cat", OneHotEncoder(handle_unknown="ignore"), CAT),
])

mean, std = cv_rmse(make_pipeline(prep_leak, LinearRegression()),
                    features=NUM_FE + ["deposit"] + CAT, X=X_leak)
print(f"с утечкой: RMSE = {mean:.2f} ± {std:.2f}   <- слишком хорошо, чтобы быть правдой")

Проверка простая: откуда этот признак возьмётся для новой квартиры, цену которой ещё никто не назвал? Взять его будет неоткуда.

### Статистика, посчитанная по всем данным

У нас есть номер дома. Соблазнительно заменить его средней ценой по дому — это называется таргет-энкодинг. Но если считать среднее по всем данным сразу, в признак попадут и те объекты, на которых мы потом проверяемся.

In [ ]:
from sklearn.model_selection import cross_val_predict

X_te = X_fe.copy()
X_te["building_mean"] = data.groupby("building")["price"].transform("mean")   # по всем данным

prep_te = ColumnTransformer([
    ("num", SimpleImputer(strategy="median"), NUM_FE + ["building_mean"]),
    ("cat", OneHotEncoder(handle_unknown="ignore"), CAT),
])

mean_leak, std_leak = cv_rmse(make_pipeline(prep_te, LinearRegression()),
                              features=NUM_FE + ["building_mean"] + CAT, X=X_te)
print(f"среднее по дому, посчитанное по всем данным: RMSE = {mean_leak:.2f} ± {std_leak:.2f}")

In [ ]:
# Честно: внутри каждого фолда среднее считается только по обучающей части.
honest = []
for train_idx, val_idx in cv.split(X_fe):
    stat = data.iloc[train_idx].groupby("building")["price"].mean()
    feat = data["building"].map(stat).fillna(y.iloc[train_idx].mean())

    X_h = X_fe.copy()
    X_h["building_mean"] = feat

    model = make_pipeline(prep_te, LinearRegression())
    model.fit(X_h.iloc[train_idx][NUM_FE + ["building_mean"] + CAT], y.iloc[train_idx])
    honest.append(rmse(y.iloc[val_idx], model.predict(X_h.iloc[val_idx][NUM_FE + ["building_mean"] + CAT])))

honest = np.array(honest)
print(f"то же самое, но честно:                     RMSE = {honest.mean():.2f} ± {honest.std():.2f}")

Разница между этими двумя числами — и есть цена утечки: первое мы бы увидели у себя, второе получили бы на проверке. Поэтому вся подготовка данных живёт внутри `Pipeline` или считается отдельно на каждом фолде.

## 9. Собираем всё вместе

Сравним линейную модель на хороших признаках и случайный лес.

In [ ]:
from sklearn.ensemble import RandomForestRegressor

ridge_final = make_pipeline(prep_fe, StandardScaler(with_mean=False), Ridge(alpha=1.0))
forest_final = make_pipeline(prep_fe, RandomForestRegressor(n_estimators=300, min_samples_leaf=2,
                                                           random_state=0, n_jobs=-1))

for name, model in [("линейная модель", ridge_final), ("случайный лес", forest_final)]:
    mean, std = cv_rmse(model, features=NUM_FE + CAT, X=X_fe)
    print(f"{name:18s} RMSE = {mean:.2f} ± {std:.2f}")

Смена модели дала меньше, чем два новых признака в разделе 5. Это обычная ситуация: сначала признаки и честная проверка, и только потом перебор моделей.

In [ ]:
# Финальная модель обучается на всех данных — и только потом идёт посылка.
best = ridge_final.fit(X_fe[NUM_FE + CAT], y)
print("модель обучена на всех данных, можно отправлять предсказания")

## Что запомнить

1. Бейзлайн — чтобы было с чем сравнивать.
2. Кросс-валидация — чтобы отличать улучшение от везения.
3. Признаки дают больше, чем смена модели.
4. Гиперпараметры подбираются на валидации, а не на тесте.
5. Слишком хороший результат — повод искать утечку, а не радоваться.
